In [32]:
from groq import Groq


client = Groq(api_key="YOUR_GROQ_API_KEY")

completion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
      {
        "role": "user",
        "content": "helllo"
      }
    ],
    temperature=1,
    max_completion_tokens=8192,
    top_p=1,
    reasoning_effort="medium",
    stream=True,
    stop=None
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")


Hello! How can I help you today?

In [33]:
from typing import TypedDict
class LLMState(TypedDict):
    query : str
    ans : str

In [34]:
def llm_qa(state : LLMState) -> LLMState :

    question = state['query']

    prompt = f'Answer the following question {question}'

    answer = ""
    completion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
      {
        "role": "user",
        "content": prompt
      }
    ],
    temperature=1,
    max_completion_tokens=8192,
    top_p=1,
    reasoning_effort="medium",
    stream=True,
    stop=None
    )
    
    for chunk in completion:
        answer += chunk.choices[0].delta.content or ""

    state['ans'] = answer

    return state
    


In [35]:
from langgraph.graph import StateGraph , END , START

graph = StateGraph(LLMState)
graph.add_node('llm_qa' , llm_qa)

graph.add_edge(START , 'llm_qa')
graph.add_edge('llm_qa' , END)


In [36]:
workflow = graph.compile()



In [37]:
initial_state = {'query' : 'what are the accomplishments of albert einsten'}

final_state = workflow.invoke(initial_state)

In [38]:
final_state

{'query': 'what are the accomplishments of albert einsten',
 'ans': '**Albert\u202fEinstein’s major accomplishments (in no particular order)**  \n\n| Year / Context | Achievement | Significance |\n|----------------|-------------|--------------|\n| **1905 – “Annus Mirabilis” papers** | • Special theory of relativity (E = mc²)<br>• Photoelectric effect (quantum hypothesis)<br>• Brownian motion (statistical physics)<br>• Mass–energy equivalence | Revolutionized space‑time, explained black‑body radiation, laid groundwork for quantum theory. |\n| **1921** | Nobel Prize in Physics (for the photoelectric effect) | First major recognition of his quantum work. |\n| **1915** | General theory of relativity | Unified gravitation with geometry of space‑time; predicted gravitational waves, bending of light, black holes, cosmological models. |\n| **1933–1940** | Development of the Einstein–de\u202fBroglie relationship, contributions to quantum statistics and Bose–Einstein condensation | Advanced stat